In [4]:
# =============================================================================
# LIGHTCONE GENERATION - MULTIPLE HII_EFF_FACTOR VALUES
# py21cmfast v3.4
# Adapted for running with HII_EFF_FACTOR = [30.0, 50.0, 70.0]
# =============================================================================

# =============================================================================
# CELL 1: Imports and Setup
# =============================================================================
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import py21cmfast as p21c
from py21cmfast import plotting

import os
from datetime import datetime

print(f"py21cmfast version: {p21c.__version__}")

# =============================================================================
# CELL 1a: Create Output Directory for Plots
# =============================================================================
plot_dir = "11DEC2025_lightcone_plots_v3/multi_HII"

# Create directory if it doesn't exist
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)
    print(f"Created directory: {plot_dir}")
else:
    print(f"Directory already exists: {plot_dir}")

print(f"All plots will be saved to: {os.path.abspath(plot_dir)}")

# =============================================================================
# Standardized Plot Settings
# =============================================================================

plt.rcParams.update({
    # Font settings
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',  # Computer Modern (LaTeX standard)
    'font.size': 20,
    'axes.labelsize': 20,
    'axes.titlesize': 20,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 16,
    'figure.titlesize': 20,
    
    # Professional ticks
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.minor.width': 0.8,
    'ytick.minor.width': 0.8,
    'xtick.top': True,
    'ytick.right': True,
    
    # Line and axes
    'axes.linewidth': 1.0,
    'lines.linewidth': 1.8,
    'lines.markersize': 5,
    
    # Grid
    'grid.linewidth': 0.5,
    'grid.alpha': 0.3,
    
    # Figure
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

# Enable minor ticks globally
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True

print("✓ Plot settings applied")

# =============================================================================
# CELL 1b: Define Parameters
# =============================================================================

# Basic simulation parameters (same for all runs)
user_params = p21c.UserParams(
    HII_DIM=128,                      # Resolution of ionization/spin-temp grids
    BOX_LEN=800.0,                    # Box size in comoving Mpc
    USE_INTERPOLATION_TABLES=True,    # Speed up calculations
    N_THREADS=8                       # Number of CPU threads
)

# Redshift range for the lightcone
z_min = 5.0                           # Final/lowest redshift
z_max = 20.0                          # Highest redshift to simulate

# **NEW: Define the HII_EFF_FACTOR values to test**
HII_EFF_FACTORS = [30.0, 50.0, 70.0]

print("User parameters defined:")
print(user_params)
print(f"\nWill run simulations for HII_EFF_FACTOR = {HII_EFF_FACTORS}")

print("\n=== DEFAULT COSMOLOGY ===")
print(p21c.CosmoParams())

print("\n=== DEFAULT ASTROPHYSICS ===")
print(p21c.AstroParams())

print("\n=== DEFAULT FLAGS ===")
print(p21c.FlagOptions())



py21cmfast version: 3.4.0
Created directory: 11DEC2025_lightcone_plots_v3/multi_HII
All plots will be saved to: /home/swanith/Desktop/Project2/Plots/Reion_hist_change_v3_lightcone/11DEC2025_lightcone_plots_v3/multi_HII
✓ Plot settings applied
User parameters defined:
UserParams:
    BOX_LEN                 : 800.0
    DIM                     : 384
    FAST_FCOLL_TABLES       : False
    HII_DIM                 : 128
    HMF                     : 1
    KEEP_3D_VELOCITIES      : False
    MINIMIZE_MEMORY         : False
    NON_CUBIC_FACTOR        : 1.0
    NO_RNG                  : False
    N_THREADS               : 8
    PERTURB_ON_HIGH_RES     : False
    POWER_SPECTRUM          : 0
    USE_2LPT                : True
    USE_FFTW_WISDOM         : False
    USE_INTERPOLATION_TABLES: True
    USE_RELATIVE_VELOCITIES : False
    

Will run simulations for HII_EFF_FACTOR = [30.0, 50.0, 70.0]

=== DEFAULT COSMOLOGY ===
CosmoParams:
    OMb        : 0.04897468161869667
    OMm        : 0.3

In [21]:
# =============================================================================
# CELL 2: Run Lightcone Simulations
# =============================================================================
# Control whether to use cached data or force recalculation
USE_CACHE = True  # Set to False to force recalculation

lightcones = {}

print("\n" + "="*70)
print("RUNNING LIGHTCONE SIMULATIONS")
print("="*70)

for hii_factor in HII_EFF_FACTORS:
    print(f"\n{'='*70}")
    print(f"HII_EFF_FACTOR = {hii_factor}")
    print(f"{'='*70}")
    
    astro_params = p21c.AstroParams(HII_EFF_FACTOR=hii_factor)
    
    print(f"Redshift range: z = {z_min} → {z_max}")
    print(f"Box size: {user_params.BOX_LEN} Mpc")
    print(f"Resolution: {user_params.HII_DIM}³ cells")
    
    lightcone = p21c.run_lightcone(
        redshift=z_min,
        max_redshift=z_max,
        lightcone_quantities=('brightness_temp', 'density', 'xH_box', 'velocity'),
        user_params=user_params,
        astro_params=astro_params,
        random_seed=37,
        direc=f'_cache_HII_{int(hii_factor)}' if USE_CACHE else None  # None forces recalc
    )
    
    lightcones[hii_factor] = lightcone
    
    print(f"\n✓ Complete! Shape: {lightcone.brightness_temp.shape}")
    print(f"  Redshift range: [{lightcone.lightcone_redshifts.min():.2f}, "
          f"{lightcone.lightcone_redshifts.max():.2f}]")

print(f"\n{'='*70}")
print(f"ALL {len(HII_EFF_FACTORS)} SIMULATIONS COMPLETE!")
print(f"{'='*70}")

# =============================================================================
# CELL 2: Plot Lightcones
# =============================================================================

print("\n" + "="*70)
print("GENERATING LIGHTCONE PLOTS")
print("="*70)

colors = {30.0: 'darkblue', 50.0: 'darkgreen', 70.0: 'darkred'}


# Stacked plots for each field
# Define custom colormaps for each field
custom_cmaps = {
    'brightness_temp': 'EoR',      # Keep default
    'xH_box': 'viridis',           # Keep default
    'density': 'magma',            # Change to magma
    'velocity': 'RdBu_r'           # Change to RdBu_r
}

for field_name, field_title in fields_to_plot:
    print(f"\nPlotting {field_name}...")
    
    fig, axes = plt.subplots(len(HII_EFF_FACTORS), 1, 
                             figsize=(12, 5*len(HII_EFF_FACTORS)), 
                             constrained_layout=True)
    
    if len(HII_EFF_FACTORS) == 1:
        axes = [axes]
    
    for idx, (hii_factor, ax) in enumerate(zip(HII_EFF_FACTORS, axes)):
        lightcone = lightcones[hii_factor]
        
        # Plot with default colormap first
        plotting.lightcone_sliceplot(lightcone, field_name, ax=ax, fig=fig)
        
        # Now change the colormap using im.set_cmap()
        im = ax.images[0]
        im.set_cmap(custom_cmaps[field_name])
        
        # Inspect what colormap was used
        cmap_used = im.get_cmap().name
        print(f"    {field_name} | HII_EFF={hii_factor}: colormap '{cmap_used}'")
        
        ax.text(0.02, 0.98, f'HII_EFF_FACTOR = {hii_factor:.1f}', 
                transform=ax.transAxes, fontsize=16, fontweight='bold',
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Save PDF without overall title
    plot_name = f"{field_name}_lightcone_multi_HII"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    
    # Add overall title for PNG
    fig.suptitle(f'{field_title} - Multiple HII_EFF_FACTOR', 
                 fontsize=24, fontweight='bold', y=0.995)
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"  ✓ Saved: {plot_name}")
    plt.close(fig)



print("\n✓ LIGHTCONE PLOTTING COMPLETE!")




RUNNING LIGHTCONE SIMULATIONS

HII_EFF_FACTOR = 30.0
Redshift range: z = 5.0 → 20.0
Box size: 800.0 Mpc
Resolution: 128³ cells

✓ Complete! Shape: (128, 128, 483)
  Redshift range: [5.00, 20.03]

HII_EFF_FACTOR = 50.0
Redshift range: z = 5.0 → 20.0
Box size: 800.0 Mpc
Resolution: 128³ cells

✓ Complete! Shape: (128, 128, 483)
  Redshift range: [5.00, 20.03]

HII_EFF_FACTOR = 70.0
Redshift range: z = 5.0 → 20.0
Box size: 800.0 Mpc
Resolution: 128³ cells

✓ Complete! Shape: (128, 128, 483)
  Redshift range: [5.00, 20.03]

ALL 3 SIMULATIONS COMPLETE!

GENERATING LIGHTCONE PLOTS

Plotting brightness_temp...
    brightness_temp | HII_EFF=30.0: colormap 'EoR'
    brightness_temp | HII_EFF=50.0: colormap 'EoR'
    brightness_temp | HII_EFF=70.0: colormap 'EoR'
  ✓ Saved: brightness_temp_lightcone_multi_HII

Plotting xH_box...
    xH_box | HII_EFF=30.0: colormap 'viridis'
    xH_box | HII_EFF=50.0: colormap 'viridis'
    xH_box | HII_EFF=70.0: colormap 'viridis'
  ✓ Saved: xH_box_lightcone_mu

In [22]:
# =============================================================================
# CELL 3: 2D SLICES AT z = 8
# =============================================================================

print("\n" + "="*70)
print("GENERATING 2D SLICES AT z = 8.0")
print("="*70)

target_z = 8.0

fields_info = [
    ('brightness_temp', '21cm Brightness Temperature [mK]', 'EoR'),
    ('xH_box', 'Neutral Fraction (xHI)', 'viridis'),
    ('density', 'Overdensity δ', 'magma'),
    ('velocity', 'Line-of-Sight Velocity [km/s]', 'RdBu_r'),
]

for field_name, field_label, cmap_name in fields_info:
    print(f"\nProcessing {field_name}...")
    
    fig, axes = plt.subplots(1, len(HII_EFF_FACTORS), 
                             figsize=(6*len(HII_EFF_FACTORS), 5.5), 
                             constrained_layout=True)
    
    if len(HII_EFF_FACTORS) == 1:
        axes = [axes]
    
    vmin_global = np.inf
    vmax_global = -np.inf
    slices_data = []
    
    for hii_factor in HII_EFF_FACTORS:
        lightcone = lightcones[hii_factor]
        z_values = lightcone.lightcone_redshifts
        closest_idx = np.argmin(np.abs(z_values - target_z))
        actual_z = z_values[closest_idx]
        
        field_data = getattr(lightcone, field_name)
        slice_2d = field_data[:, :, closest_idx]
        
        slices_data.append((slice_2d, actual_z))
        vmin_global = min(vmin_global, slice_2d.min())
        vmax_global = max(vmax_global, slice_2d.max())
    
    for idx, (hii_factor, ax) in enumerate(zip(HII_EFF_FACTORS, axes)):
        slice_2d, actual_z = slices_data[idx]
        
        im = ax.imshow(slice_2d.T, origin='lower', cmap=cmap_name,
                      vmin=vmin_global, vmax=vmax_global,
                      extent=[0, user_params.BOX_LEN, 0, user_params.BOX_LEN],
                      aspect='auto')
        
        ax.set_xlabel('Comoving Distance [Mpc]', fontsize=14)
        ax.set_ylabel('Comoving Distance [Mpc]', fontsize=14)
        ax.set_title(f'HII_EFF = {hii_factor:.1f} (z = {actual_z:.2f})', 
                    fontsize=16, fontweight='bold')
        
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=11)
    
    fig.suptitle(f'{field_label} at z ≈ {target_z}', 
                fontsize=20, fontweight='bold')
    
    plot_name = f"{field_name}_slice_z{int(target_z)}_multi_HII"
    fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
    fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
    
    print(f"  ✓ Saved: {plot_name}")
    print(f"    Range: [{vmin_global:.3f}, {vmax_global:.3f}]")
    plt.close(fig)

print("\n✓ 2D SLICE PLOTTING COMPLETE!")


GENERATING 2D SLICES AT z = 8.0

Processing brightness_temp...
  ✓ Saved: brightness_temp_slice_z8_multi_HII
    Range: [0.000, 18.834]

Processing xH_box...
  ✓ Saved: xH_box_slice_z8_multi_HII
    Range: [0.000, 0.941]

Processing density...
  ✓ Saved: density_slice_z8_multi_HII
    Range: [-0.388, 0.744]

Processing velocity...
  ✓ Saved: velocity_slice_z8_multi_HII
    Range: [-0.000, 0.000]

✓ 2D SLICE PLOTTING COMPLETE!


In [10]:
# =============================================================================
# CELL 3b: Summary Statistics for 2D Slices at z=8
# =============================================================================

print("\n" + "="*70)
print(f"SUMMARY STATISTICS FOR 2D SLICES AT z ≈ {target_z}")
print("="*70)

for hii_factor in HII_EFF_FACTORS:
    lightcone = lightcones[hii_factor]
    
    # Find the closest redshift slice
    z_values = lightcone.lightcone_redshifts
    closest_idx = np.argmin(np.abs(z_values - target_z))
    actual_z = z_values[closest_idx]
    
    # Extract 2D slices
    brightness_slice = lightcone.brightness_temp[:, :, closest_idx]
    xHI_slice = lightcone.xH_box[:, :, closest_idx]
    density_slice = lightcone.density[:, :, closest_idx]  # δ
    velocity_slice = 1e17*lightcone.velocity[:, :, closest_idx]
    
    print(f"\nHII_EFF_FACTOR = {hii_factor} (z = {actual_z:.2f}):")
    print(f"  Brightness temp [mK]: min={brightness_slice.min():.2f}, "
          f"max={brightness_slice.max():.2f}, mean={brightness_slice.mean():.2f}")
    print(f"  Neutral fraction:     min={xHI_slice.min():.4f}, "
          f"max={xHI_slice.max():.4f}, mean={xHI_slice.mean():.4f}")
    print(f"  Overdensity δ:        min={density_slice.min():.3f}, "
          f"max={density_slice.max():.3f}, mean={density_slice.mean():.3f}")
    print(f"  Velocity 1e-17 [Mpc/s]:      min={velocity_slice.min():.2f}, "
          f"max={velocity_slice.max():.2f}, mean={velocity_slice.mean():.2f}")

print("\n" + "="*70)


# =============================================================================
# CELL 3c: Summary Statistics for Full Lightcones
# =============================================================================

print("\n" + "="*70)
print("SUMMARY STATISTICS FOR FULL LIGHTCONES")
print("="*70)

for hii_factor in HII_EFF_FACTORS:
    lightcone = lightcones[hii_factor]
    
    print(f"\nHII_EFF_FACTOR = {hii_factor}:")
    print(f"  Redshift range: [{lightcone.lightcone_redshifts.min():.2f}, "
          f"{lightcone.lightcone_redshifts.max():.2f}]")
    
    # Full 3D field statistics
    print(f"  Brightness temp [mK]: min={lightcone.brightness_temp.min():.2f}, "
          f"max={lightcone.brightness_temp.max():.2f}, "
          f"mean={lightcone.brightness_temp.mean():.2f}")
    print(f"  Neutral fraction:     min={lightcone.xH_box.min():.4f}, "
          f"max={lightcone.xH_box.max():.4f}, "
          f"mean={lightcone.xH_box.mean():.4f}")
    print(f"  Overdensity δ:        min={lightcone.density.min():.3f}, "
          f"max={lightcone.density.max():.3f}, "
          f"mean={lightcone.density.mean():.3f}")
    print(f"  Velocity [1e-17 Mpc/s]:      min={lightcone.velocity.min():.2f}, "
          f"max={lightcone.velocity.max():.2f}, "
          f"mean={lightcone.velocity.mean():.2f}")

print("\n" + "="*70)


SUMMARY STATISTICS FOR 2D SLICES AT z ≈ 8.0

HII_EFF_FACTOR = 30.0 (z = 8.01):
  Brightness temp [mK]: min=0.00, max=18.83, mean=11.43
  Neutral fraction:     min=0.0000, max=0.9412, mean=0.4990
  Overdensity δ:        min=-0.388, max=0.744, mean=-0.005
  Velocity 1e-17 [Mpc/s]:      min=-11.84, max=14.21, mean=0.02

HII_EFF_FACTOR = 50.0 (z = 8.01):
  Brightness temp [mK]: min=0.00, max=15.67, mean=5.58
  Neutral fraction:     min=0.0000, max=0.9020, mean=0.2626
  Overdensity δ:        min=-0.388, max=0.744, mean=-0.005
  Velocity 1e-17 [Mpc/s]:      min=-11.84, max=14.21, mean=0.02

HII_EFF_FACTOR = 70.0 (z = 8.01):
  Brightness temp [mK]: min=0.00, max=14.06, mean=2.45
  Neutral fraction:     min=0.0000, max=0.8628, mean=0.1230
  Overdensity δ:        min=-0.388, max=0.744, mean=-0.005
  Velocity 1e-17 [Mpc/s]:      min=-11.84, max=14.21, mean=0.02


SUMMARY STATISTICS FOR FULL LIGHTCONES

HII_EFF_FACTOR = 30.0:
  Redshift range: [5.00, 20.03]
  Brightness temp [mK]: min=0.00, max=

In [19]:
# =============================================================================
# CELL 4: Reionization History Overlay
# =============================================================================

print("\n" + "="*70)
print("GENERATING REIONIZATION HISTORY COMPARISON")
print("="*70)

colors_map = {30.0: 'darkblue', 50.0: 'darkgreen', 70.0: 'darkred'}
linestyles_map = {30.0: '-', 50.0: '--', 70.0: '-.'}

# Ionization fraction vs redshift
fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for hii_factor in HII_EFF_FACTORS:
    lightcone = lightcones[hii_factor]
    z_nodes = lightcone.node_redshifts[::-1]
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    ax.plot(z_nodes, x_e_nodes, 
           linestyle=linestyles_map[hii_factor],
           linewidth=2.5, color=colors_map[hii_factor],
           label=f'HII_EFF = {hii_factor:.1f}',
           marker='o', markersize=4, alpha=0.8)

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Ionization Fraction $x_e$', fontsize=20)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='best', fontsize=16, framealpha=0.9)
ax.invert_xaxis()
#ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "reionization_history_xe_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Reionization History: Ionization Fraction', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# Neutral fraction vs redshift
fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for hii_factor in HII_EFF_FACTORS:
    lightcone = lightcones[hii_factor]
    z_nodes = lightcone.node_redshifts[::-1]
    xHI_nodes = lightcone.global_xH[::-1]
    
    ax.plot(z_nodes, xHI_nodes, 
           linestyle=linestyles_map[hii_factor],
           linewidth=2.5, color=colors_map[hii_factor],
           label=f'HII_EFF = {hii_factor:.1f}',
           marker='o', markersize=4, alpha=0.8)

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Neutral Fraction $x_{\rm HI}$', fontsize=20)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='best', fontsize=16, framealpha=0.9)
ax.invert_xaxis()
#ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "reionization_history_xHI_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Reionization History: Neutral Fraction', 
            fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# Compute milestones
print("\nReionization Milestones:")
print("-" * 70)

milestones_data = {}

for hii_factor in HII_EFF_FACTORS:
    lightcone = lightcones[hii_factor]
    z_nodes = lightcone.node_redshifts[::-1]
    x_e_nodes = 1.0 - lightcone.global_xH[::-1]
    
    idx_50 = np.argmin(np.abs(x_e_nodes - 0.5))
    z_50 = z_nodes[idx_50]
    
    idx_90 = np.argmin(np.abs(x_e_nodes - 0.9))
    z_90 = z_nodes[idx_90]
    
    idx_10 = np.argmin(np.abs(x_e_nodes - 0.1))
    z_10 = z_nodes[idx_10]
    
    delta_z = z_10 - z_90
    
    milestones_data[hii_factor] = {
        'z_10': z_10, 'z_50': z_50, 'z_90': z_90, 'delta_z': delta_z
    }
    
    print(f"HII_EFF_FACTOR = {hii_factor:.1f}:")
    print(f"  z at 10% ionized: {z_10:.2f}")
    print(f"  z at 50% ionized: {z_50:.2f}")
    print(f"  z at 90% ionized: {z_90:.2f}")
    print(f"  Δz (10% → 90%):   {delta_z:.2f}")

# Plot milestones
fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

hii_vals = list(milestones_data.keys())
z_10_vals = [milestones_data[h]['z_10'] for h in hii_vals]
z_50_vals = [milestones_data[h]['z_50'] for h in hii_vals]
z_90_vals = [milestones_data[h]['z_90'] for h in hii_vals]

x_pos = np.arange(len(hii_vals))
width = 0.25

ax.bar(x_pos - width, z_10_vals, width, label='10% ionized', 
       color='lightblue', edgecolor='black', linewidth=1.5)
ax.bar(x_pos, z_50_vals, width, label='50% ionized', 
       color='lightgreen', edgecolor='black', linewidth=1.5)
ax.bar(x_pos + width, z_90_vals, width, label='90% ionized', 
       color='lightcoral', edgecolor='black', linewidth=1.5)

ax.set_xlabel('HII_EFF_FACTOR', fontsize=20)
ax.set_ylabel('Redshift', fontsize=20)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{h:.1f}' for h in hii_vals])
ax.legend(loc='best', fontsize=16)
#ax.grid(True, axis='y', alpha=0.3, linestyle='--')

plot_name = "reionization_milestones_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Reionization Milestones', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# Duration plot
fig, ax = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)

delta_z_vals = [milestones_data[h]['delta_z'] for h in hii_vals]
colors_list = [colors_map[h] for h in hii_vals]

bars = ax.bar(x_pos, delta_z_vals, color=colors_list, 
              edgecolor='black', linewidth=1.5, alpha=0.7)

ax.set_xlabel('HII_EFF_FACTOR', fontsize=20)
ax.set_ylabel(r'Duration $\Delta z$ (10% → 90%)', fontsize=20)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{h:.1f}' for h in hii_vals])
#ax.grid(True, axis='y', alpha=0.3, linestyle='--')

for bar, val in zip(bars, delta_z_vals):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{val:.2f}',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

plot_name = "reionization_duration_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Reionization Duration', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n✓ REIONIZATION HISTORY COMPLETE!")

# =============================================================================
# Final Summary
# =============================================================================

print("\n" + "="*70)
print("ALL ANALYSIS COMPLETE!")
print("="*70)
print(f"\nAnalyzed HII_EFF_FACTOR values: {HII_EFF_FACTORS}")
print(f"\nGenerated:")
print(f"  • Lightcone plots for all fields")
print(f"  • 2D slices at z ≈ {target_z}")
print(f"  • Reionization history comparisons")
print(f"  • Milestone and duration analysis")
print(f"\nAll plots saved to: {os.path.abspath(plot_dir)}")
print("="*70)



GENERATING REIONIZATION HISTORY COMPARISON
✓ Saved: reionization_history_xe_multi_HII
✓ Saved: reionization_history_xHI_multi_HII

Reionization Milestones:
----------------------------------------------------------------------
HII_EFF_FACTOR = 30.0:
  z at 10% ionized: 11.00
  z at 50% ionized: 8.09
  z at 90% ionized: 6.31
  Δz (10% → 90%):   4.69
HII_EFF_FACTOR = 50.0:
  z at 10% ionized: 11.73
  z at 50% ionized: 9.04
  z at 90% ionized: 7.40
  Δz (10% → 90%):   4.33
HII_EFF_FACTOR = 70.0:
  z at 10% ionized: 12.25
  z at 50% ionized: 9.45
  z at 90% ionized: 7.92
  Δz (10% → 90%):   4.33
✓ Saved: reionization_milestones_multi_HII
✓ Saved: reionization_duration_multi_HII

✓ REIONIZATION HISTORY COMPLETE!

ALL ANALYSIS COMPLETE!

Analyzed HII_EFF_FACTOR values: [30.0, 50.0, 70.0]

Generated:
  • Lightcone plots for all fields
  • 2D slices at z ≈ 8.0
  • Reionization history comparisons
  • Milestone and duration analysis

All plots saved to: /home/swanith/Desktop/Project2/Plots/Rei

In [18]:
# =============================================================================
# CELL 4f: Optical Depth Calculations for All HII_EFF_FACTOR Values
# =============================================================================

print("\n" + "="*70)
print("OPTICAL DEPTH CALCULATIONS")
print("="*70)

# Physical constants for optical depth calculation
c_km_s = 2.998e5                    # Speed of light [km/s]
h = 0.6766                          # Hubble parameter (from default cosmology)
H0 = 100 * h                        # Hubble constant [km/s/Mpc]
Omega_b = 0.04897468161869667       # Baryon density (from default cosmology)
Omega_m = 0.30964144154550644       # Matter density (from default cosmology)

# Critical density of the universe [protons/cm^3]
rho_crit_p_cm3 = 1.88e-29 * h**2 / (1.67e-24)  # Convert to protons/cm^3
n_H0_cm3 = Omega_b * rho_crit_p_cm3             # Mean hydrogen number density [cm^-3]

# Thomson scattering cross section
sigma_T_cm2 = 6.65e-25              # [cm^2]

# Convert to Mpc units
cm_per_Mpc = 3.086e24
n_e0_Mpc3 = n_H0_cm3 * cm_per_Mpc**3
sigma_T_Mpc2 = sigma_T_cm2 / cm_per_Mpc**2

# Prefactor for dτ calculation
prefactor = n_e0_Mpc3 * sigma_T_Mpc2  # [Mpc^-1]

print(f"\nPhysical constants:")
print(f"  n_H0 = {n_H0_cm3:.6e} cm^-3")
print(f"  σ_T = {sigma_T_cm2:.6e} cm^2")
print(f"  Prefactor = {prefactor:.6e} Mpc^-1")

# Dictionary to store optical depth results
tau_results = {}

for hii_factor in HII_EFF_FACTORS:
    lightcone = lightcones[hii_factor]
    
    # Extract redshift and distance axes
    red_axis = lightcone.lightcone_redshifts
    pos_axis = lightcone.lightcone_distances  # Comoving distance [Mpc]
    
    # Trim to z <= z_max if needed
    ind_z = np.where(red_axis <= z_max)[0]
    red_axis = red_axis[ind_z]
    pos_axis = pos_axis[ind_z]
    
    # Get ionization history
    z_nodes_sorted = lightcone.node_redshifts[::-1]
    xHI_nodes_sorted = lightcone.global_xH[::-1]
    x_e_nodes_sorted = 1.0 - xHI_nodes_sorted
    
    # Interpolate x_e onto lightcone redshift grid
    x_e_interp = np.interp(red_axis, z_nodes_sorted, x_e_nodes_sorted)
    
    # Calculate geometric quantities
    s = pos_axis  # Comoving distance [Mpc]
    ds = np.diff(s)  # Distance element [Mpc]
    
    # Midpoint values for integration
    z_mid = 0.5 * (red_axis[:-1] + red_axis[1:])
    x_e_mid = 0.5 * (x_e_interp[:-1] + x_e_interp[1:])
    
    # Calculate dτ
    dtau = prefactor * x_e_mid * (1.0 + z_mid)**2 * ds
    
    # Cumulative optical depth
    tau = np.cumsum(dtau)
    tau_total = tau[-1]
    
    # Store results
    tau_results[hii_factor] = {
        'red_axis': red_axis,
        'pos_axis': pos_axis,
        's': s,
        'ds': ds,
        'z_mid': z_mid,
        'x_e_interp': x_e_interp,
        'x_e_mid': x_e_mid,
        'dtau': dtau,
        'tau': tau,
        'tau_total': tau_total
    }
    
    print(f"\nHII_EFF_FACTOR = {hii_factor}:")
    print(f"  Redshift range:     {red_axis.min():.2f} → {red_axis.max():.2f}")
    print(f"  Comoving distance:  {s.min():.1f} → {s.max():.1f} Mpc")
    print(f"  Mean ds:            {ds.mean():.3f} Mpc")
    print(f"  x_e range:          {x_e_interp.min():.4f} → {x_e_interp.max():.4f}")
    print(f"  dτ range:           {dtau.min():.6e} → {dtau.max():.6e}")
    print(f"  Total τ:            {tau_total:.6f}")

print("\n" + "="*70)

# =============================================================================
# PLOT: Comoving Distance s vs z (Overlay)
# =============================================================================


fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for hii_factor in HII_EFF_FACTORS:
    results = tau_results[hii_factor]
    
    # Strip units before plotting
    red_axis_plot = np.asarray(results['red_axis'])
    s_plot = np.asarray(results['s'])
    
    ax.plot(red_axis_plot, s_plot, 
           linestyle=linestyles_map[hii_factor],
           linewidth=2.5, 
           color=colors_map[hii_factor],
           label=f'HII_EFF = {hii_factor:.1f}')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel('Comoving Distance $s$ [Mpc]', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.invert_xaxis()
#ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "s_vs_z_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Comoving Distance vs Redshift', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Distance Element ds vs z (Overlay)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for hii_factor in HII_EFF_FACTORS:
    results = tau_results[hii_factor]
    
    # Strip units before plotting
    z_mid_plot = np.asarray(results['z_mid'])
    ds_plot = np.asarray(results['ds'])
    
    ax.plot(z_mid_plot, ds_plot, 
           linestyle=linestyles_map[hii_factor],
           linewidth=2.5, 
           color=colors_map[hii_factor],
           label=f'HII_EFF = {hii_factor:.1f}',
           marker='o',
           markersize=3,
           alpha=0.8)

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel('Distance Element $ds$ [Mpc]', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.invert_xaxis()
#ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "ds_vs_z_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Distance Element vs Redshift', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Optical Depth Element dτ vs z (Overlay)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for hii_factor in HII_EFF_FACTORS:
    results = tau_results[hii_factor]
    
    # Strip units before plotting
    z_mid_plot = np.asarray(results['z_mid'])
    dtau_plot = np.asarray(results['dtau'])
    
    ax.plot(z_mid_plot, dtau_plot, 
           linestyle=linestyles_map[hii_factor],
           linewidth=2.5, 
           color=colors_map[hii_factor],
           label=f'HII_EFF = {hii_factor:.1f}',
           marker='o',
           markersize=3,
           alpha=0.8)

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Optical Depth Element $d\tau$', fontsize=20)
ax.legend(loc='best', fontsize=16)
ax.invert_xaxis()
#ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "dtau_vs_z_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title(r'Optical Depth Element $d\tau$ vs Redshift', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)


# =============================================================================
# PLOT: Cumulative Optical Depth τ vs z (Overlay)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for hii_factor in HII_EFF_FACTORS:
    results = tau_results[hii_factor]
    
    # Strip units before plotting
    z_mid_plot = np.asarray(results['z_mid'])
    tau_plot = np.asarray(results['tau'])
    tau_total_plot = float(np.asarray(results['tau_total']))
    
    ax.plot(z_mid_plot, tau_plot, 
           linestyle=linestyles_map[hii_factor],
           linewidth=2.5, 
           color=colors_map[hii_factor],
           label=f'HII_EFF = {hii_factor:.1f} (τ = {tau_total_plot:.4f})',
           marker='o',
           markersize=3,
           alpha=0.8)

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Cumulative Optical Depth $\tau(<z)$', fontsize=20)
ax.legend(loc='best', fontsize=14)
ax.invert_xaxis()
#ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "tau_vs_z_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Cumulative Optical Depth vs Redshift', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# PLOT: Total Optical Depth Comparison (Bar Chart)
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(8, 6), constrained_layout=True)

hii_vals = list(HII_EFF_FACTORS)
# Strip units before plotting
tau_total_vals = [float(np.asarray(tau_results[h]['tau_total'])) for h in hii_vals]
colors_list = [colors_map[h] for h in hii_vals]

x_pos = np.arange(len(hii_vals))
bars = ax.bar(x_pos, tau_total_vals, color=colors_list, 
              edgecolor='black', linewidth=1.5, alpha=0.7)

ax.set_xlabel('HII_EFF_FACTOR', fontsize=20)
ax.set_ylabel(r'Total Optical Depth $\tau$', fontsize=20)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{h:.1f}' for h in hii_vals])
#ax.grid(True, axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bar, val in zip(bars, tau_total_vals):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}',
            ha='center', va='bottom', fontsize=14, fontweight='bold')

plot_name = "tau_total_comparison_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')
ax.set_title('Total Optical Depth Comparison', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')
print(f"✓ Saved: {plot_name}")
plt.close(fig)

print("\n" + "="*70)
print("OPTICAL DEPTH ANALYSIS COMPLETE!")
print("="*70)


OPTICAL DEPTH CALCULATIONS

Physical constants:
  n_H0 = 2.523928e-07 cm^-3
  σ_T = 6.650000e-25 cm^2
  Prefactor = 5.179580e-07 Mpc^-1

HII_EFF_FACTOR = 30.0:
  Redshift range:     5.00 → 19.95
  Comoving distance:  7946.5 Mpc → 10952.7 Mpc Mpc
  Mean ds:            6.250 Mpc Mpc
  x_e range:          0.0000 → 0.9992
  dτ range:           7.458399e-08 Mpc → 1.605852e-04 Mpc
  Total τ:            0.040152 Mpc

HII_EFF_FACTOR = 50.0:
  Redshift range:     5.00 → 19.95
  Comoving distance:  7946.5 Mpc → 10952.7 Mpc Mpc
  Mean ds:            6.250 Mpc Mpc
  x_e range:          0.0001 → 1.0000
  dτ range:           1.244832e-07 Mpc → 2.030308e-04 Mpc
  Total τ:            0.051520 Mpc

HII_EFF_FACTOR = 70.0:
  Redshift range:     5.00 → 19.95
  Comoving distance:  7946.5 Mpc → 10952.7 Mpc Mpc
  Mean ds:            6.250 Mpc Mpc
  x_e range:          0.0001 → 1.0000
  dτ range:           1.743000e-07 Mpc → 2.327453e-04 Mpc
  Total τ:            0.058991 Mpc

✓ Saved: s_vs_z_multi_HII
✓ Sav

In [ ]:
# =============================================================================
# CELL 5: Compute kSZ Integrand with Visibility Function for Multiple HII_EFF_FACTOR
# kSZ integrand = (1 + δ) × x_e × v_z / c × e^(-τ(z))
# =============================================================================

print("\n" + "="*70)
print("COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION")
print("="*70)

# Speed of light in Mpc/s
c_Mpc_s = 299792.458 / 3.08567758e19  # Mpc/s
print(f"Speed of light: c = {c_Mpc_s:.6e} Mpc/s")

# Dictionary to store kSZ results
kSZ_results = {}

for hii_factor in HII_EFF_FACTORS:
    print(f"\n{'='*70}")
    print(f"HII_EFF_FACTOR = {hii_factor}")
    print(f"{'='*70}")
    
    lightcone = lightcones[hii_factor]
    results = tau_results[hii_factor]
    
    # Extract redshift and distance axes (strip units)
    red_axis = np.asarray(results['red_axis'])
    z_mid = np.asarray(results['z_mid'])
    tau = np.asarray(results['tau'])
    
    # Extract 3D fields
    # Find indices for trimmed redshift range
    red_axis_full = np.asarray(lightcone.lightcone_redshifts)
    ind_z = np.where(red_axis_full <= z_max)[0]
    
    density_1plus = 1 + np.asarray(lightcone.density[:, :, ind_z])  # 1 + δ
    x_e_3D = 1 - np.asarray(lightcone.xH_box[:, :, ind_z])          # Ionized fraction
    v_los_Mpc_s = np.asarray(lightcone.velocity[:, :, ind_z])       # Velocity [Mpc/s]
    
    print(f"3D field shapes: {density_1plus.shape}")
    
    # =============================================================================
    # Interpolate τ(z) onto lightcone redshifts
    # =============================================================================
    
    # tau is defined at z_mid (midpoints), need to interpolate to red_axis
    # Extend tau to match red_axis length by prepending 0 (at lowest z)
    tau_extended = np.concatenate([[0], tau])
    
    # Interpolate tau onto red_axis
    tau_at_lightcone = np.interp(red_axis, 
                                  np.concatenate([[red_axis[0]], z_mid]), 
                                  tau_extended)
    
    print(f"τ range: [{tau_at_lightcone.min():.6f}, {tau_at_lightcone.max():.6f}]")
    
    # Visibility function e^(-τ)
    visibility = np.exp(-tau_at_lightcone)
    
    print(f"e^(-τ) range: [{visibility.min():.6f}, {visibility.max():.6f}]")
    
    # Broadcast visibility to 3D for multiplication with lightcone fields
    visibility_3D = visibility[None, None, :]  # Shape (1, 1, n_redshift)
    
    # =============================================================================
    # Compute kSZ integrand WITH visibility function
    # =============================================================================
    
    kSZ_integrand = density_1plus * x_e_3D * v_los_Mpc_s / c_Mpc_s * visibility_3D
    
    print(f"\nkSZ INTEGRAND (with visibility) STATISTICS:")
    print(f"  Mean: {kSZ_integrand.mean():.4e}")
    print(f"  Std:  {kSZ_integrand.std():.4e}")
    print(f"  Min:  {kSZ_integrand.min():.4e}")
    print(f"  Max:  {kSZ_integrand.max():.4e}")
    print(f"  RMS:  {np.sqrt(np.mean(kSZ_integrand**2)):.4e}")
    
    # Store results
    kSZ_results[hii_factor] = {
        'kSZ_integrand': kSZ_integrand,
        'visibility': visibility,
        'visibility_3D': visibility_3D,
        'tau_at_lightcone': tau_at_lightcone,
        'red_axis': red_axis,
        'ind_z': ind_z
    }
    
    # Add to lightcone object for easy plotting
    lightcone.kSZ_integrand = kSZ_integrand
    lightcone.visibility_func = visibility_3D

print("\n" + "="*70)
print("kSZ INTEGRAND CALCULATION COMPLETE")
print("="*70)

# =============================================================================
# PLOT: kSZ Integrand WITH Visibility - All HII_EFF_FACTOR Values
# =============================================================================

print("\n" + "="*70)
print("GENERATING kSZ INTEGRAND PLOTS")
print("="*70)

# Stacked plots
fig, axes = plt.subplots(len(HII_EFF_FACTORS), 1, 
                         figsize=(12, 5*len(HII_EFF_FACTORS)), 
                         constrained_layout=True)

if len(HII_EFF_FACTORS) == 1:
    axes = [axes]

for idx, (hii_factor, ax) in enumerate(zip(HII_EFF_FACTORS, axes)):
    lightcone = lightcones[hii_factor]
    
    plotting.lightcone_sliceplot(lightcone, 'kSZ_integrand', ax=ax, fig=fig)
    
    # Change colormap to seismic (better for data centered near zero)
    im = ax.images[0]
    im.set_cmap('seismic')
    
    # Set symmetric color limits around zero
    kSZ_data = kSZ_results[hii_factor]['kSZ_integrand']
    vmax = np.percentile(np.abs(kSZ_data), 99)  # Use 99th percentile
    im.set_clim(-vmax, vmax)
    
    # Add label
    ax.text(0.02, 0.98, f'HII_EFF_FACTOR = {hii_factor:.1f}', 
            transform=ax.transAxes, 
            fontsize=16, fontweight='bold',
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Save
plot_name = "kSZ_integrand_with_visibility_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

fig.suptitle(r'kSZ Integrand: $(1+\delta) \times x_e \times v_z/c \times e^{-\tau(z)}$', 
             fontsize=24, fontweight='bold', y=0.995)
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)



# =============================================================================
# PLOT: Visibility Function e^(-τ) vs z
# =============================================================================

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)

for hii_factor in HII_EFF_FACTORS:
    results = kSZ_results[hii_factor]
    red_axis = results['red_axis']
    visibility = results['visibility']
    
    ax.plot(red_axis, visibility, 
           linestyle=linestyles_map[hii_factor],
           linewidth=2.5, 
           color=colors_map[hii_factor],
           label=f'HII_EFF = {hii_factor:.1f}')

ax.set_xlabel('Redshift $z$', fontsize=20)
ax.set_ylabel(r'Visibility Function $e^{-\tau(z)}$', fontsize=20)
ax.set_ylim(0, 1.05)
ax.legend(loc='best', fontsize=16)
ax.invert_xaxis()
#ax.grid(True, alpha=0.3, linestyle='--')

plot_name = "visibility_function_multi_HII"
fig.savefig(f"{plot_dir}/{plot_name}.pdf", bbox_inches='tight')

ax.set_title(r'Visibility Function $e^{-\tau(z)}$', fontsize=22, fontweight='bold')
fig.savefig(f"{plot_dir}/{plot_name}.png", dpi=300, bbox_inches='tight')

print(f"✓ Saved: {plot_name}")
plt.close(fig)

# =============================================================================
# Summary Statistics
# =============================================================================

print("\n" + "="*70)
print("kSZ INTEGRAND SUMMARY")
print("="*70)

for hii_factor in HII_EFF_FACTORS:
    kSZ_data = kSZ_results[hii_factor]['kSZ_integrand']
    rms = np.sqrt(np.mean(kSZ_data**2))
    
    print(f"\nHII_EFF_FACTOR = {hii_factor}:")
    print(f"  kSZ RMS:         {rms:.4e}")
    print(f"  kSZ Mean:        {kSZ_data.mean():.4e}")
    print(f"  kSZ Std:         {kSZ_data.std():.4e}")
    print(f"  Min visibility:  {kSZ_results[hii_factor]['visibility'].min():.4f}")
    print(f"  Max visibility:  {kSZ_results[hii_factor]['visibility'].max():.4f}")

print("\n" + "="*70)
print("kSZ ANALYSIS COMPLETE!")
print("="*70)


COMPUTING kSZ INTEGRAND WITH VISIBILITY FUNCTION
Speed of light: c = 9.715612e-15 Mpc/s

HII_EFF_FACTOR = 30.0
3D field shapes: (128, 128, 482)
τ range: [0.000000, 0.040152]
e^(-τ) range: [0.960644, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 1.6627e-06
  Std:  2.1593e-03
  Min:  -2.3798e-02
  Max:  2.7095e-02
  RMS:  2.1593e-03

HII_EFF_FACTOR = 50.0
3D field shapes: (128, 128, 482)
τ range: [0.000000, 0.051520]
e^(-τ) range: [0.949785, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 1.1617e-06
  Std:  2.4099e-03
  Min:  -2.3792e-02
  Max:  2.7094e-02
  RMS:  2.4099e-03

HII_EFF_FACTOR = 70.0
3D field shapes: (128, 128, 482)
τ range: [0.000000, 0.058991]
e^(-τ) range: [0.942716, 1.000000]

kSZ INTEGRAND (with visibility) STATISTICS:
  Mean: 3.6240e-07
  Std:  2.5473e-03
  Min:  -2.3792e-02
  Max:  2.7094e-02
  RMS:  2.5473e-03

kSZ INTEGRAND CALCULATION COMPLETE

GENERATING kSZ INTEGRAND PLOTS
✓ Saved: kSZ_integrand_with_visibility_multi_HII
✓ Saved: